# Паттерн 10: Debate — дебаты с медиатором

Debate — это паттерн мультиагентной системы, в котором два агента с противоположными позициями (оптимист и пессимист) спорят о заданном вопросе в нескольких раундах. После завершения дебатов медиатор синтезирует сбалансированное заключение, учитывая сильные стороны обеих позиций.

Поле `optimist_args` и `pessimist_args` используют `operator.add` — аргументы всех раундов накапливаются. Каждый участник видит историю дебатов и может отвечать на аргументы оппонента. Счётчик `round` управляет циклом: после заданного числа раундов управление переходит к медиатору.

```mermaid
graph LR
    START((START)) --> optimist(["🔹 Optimist<br/><i>аргументирует позицию</i>"])
    optimist --> pessimist(["🔹 Pessimist<br/><i>аргументирует позицию</i>"])
    pessimist --> counter(["🎯 Counter<br/><i>управляет раундами</i>"])
    counter -->|next_round| optimist
    counter -->|done| mediator(["📊 Mediator<br/><i>выносит вердикт</i>"])
    mediator --> END((END))

    classDef terminal fill:#95A5A6,stroke:#707B7C,color:#fff
    classDef worker fill:#2ECC71,stroke:#1A8B4C,color:#fff,stroke-width:2px
    classDef coordinator fill:#4A90D9,stroke:#2C5F8A,color:#fff,stroke-width:2px
    classDef aggregator fill:#F59E0B,stroke:#D97706,color:#fff,stroke-width:2px

    class START,END terminal
    class optimist,pessimist worker
    class counter coordinator
    class mediator aggregator
```

In [1]:
import sys
sys.path.insert(0, "..")

import operator
from typing import TypedDict, Annotated

from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage
from llm_config import get_llm, check_api_key

In [2]:
if not check_api_key():
    raise ValueError("API key is not set")
else:
    print("API key is set")

API key is set


## Состояние дебатов

Состояние `DebateState` содержит пять полей. Поля `optimist_args` и `pessimist_args` используют редьюсер `operator.add` — аргументы всех раундов накапливаются автоматически, и каждый участник видит полную историю дебатов. Поле `round` отслеживает текущий раунд, а `conclusion` заполняется медиатором в самом конце.

In [3]:
llm = get_llm()

class DebateState(TypedDict):
    question: str
    optimist_args: Annotated[list[str], operator.add]
    pessimist_args: Annotated[list[str], operator.add]
    round: int
    conclusion: str

## Участники дебатов: оптимист и пессимист

Каждый участник имеет узкую роль: оптимист ищет возможности и преимущества, пессимист — риски и угрозы. Оба получают контекст аргументов оппонента из предыдущих раундов и должны на них ответить. Оптимист также увеличивает счётчик раундов при каждом вызове.

In [4]:
def optimist(state: DebateState) -> dict:
    """Агент-оптимист: ищет возможности и преимущества."""
    current_round = state.get("round", 0) + 1

    # Контекст предыдущих аргументов оппонента
    opponent_context = ""
    if state["pessimist_args"]:
        opponent_context = f"\n\nАргументы пессимиста (ответь на них):\n" + "\n".join(state["pessimist_args"])

    response = llm.invoke([
        SystemMessage(content="""Ты — оптимист-аналитик. Ищи возможности, преимущества, позитивные сценарии.
Если оппонент привёл контраргументы — ответь на них и приведи свои новые аргументы.
Будь убедителен, но честен. Кратко: 2-3 аргумента."""),
        HumanMessage(content=f"Вопрос: {state['question']}{opponent_context}")
    ])

    print(f"  [Оптимист, раунд {current_round}] {response.content[:80]}...")
    return {
        "optimist_args": [f"Раунд {current_round}: {response.content}"],
        "round": current_round,
    }


def pessimist(state: DebateState) -> dict:
    """Агент-пессимист: ищет риски и угрозы."""
    current_round = state["round"]

    # Контекст аргументов оптимиста
    opponent_context = ""
    if state["optimist_args"]:
        opponent_context = f"\n\nАргументы оптимиста (ответь на них):\n" + "\n".join(state["optimist_args"])

    response = llm.invoke([
        SystemMessage(content="""Ты — пессимист-аналитик. Ищи риски, угрозы, негативные сценарии.
Если оппонент привёл аргументы — ответь на них и приведи свои контраргументы.
Будь убедителен, но честен. Кратко: 2-3 аргумента."""),
        HumanMessage(content=f"Вопрос: {state['question']}{opponent_context}")
    ])

    print(f"  [Пессимист, раунд {current_round}] {response.content[:80]}...")
    return {"pessimist_args": [f"Раунд {current_round}: {response.content}"]}

## Медиатор

Медиатор вступает в игру после завершения всех раундов. Он получает полные списки аргументов обеих сторон и синтезирует сбалансированное заключение. Его задача — учесть сильные стороны обеих позиций и дать конкретную рекомендацию.

In [5]:
def mediator(state: DebateState) -> dict:
    """Медиатор: синтезирует сбалансированный вывод из дебатов."""
    opt_args = "\n".join(state["optimist_args"])
    pess_args = "\n".join(state["pessimist_args"])

    response = llm.invoke([
        SystemMessage(content="""Ты — медиатор. Изучи аргументы обеих сторон и синтезируй
сбалансированное заключение. Учти сильные стороны обоих позиций.
Дай конкретную рекомендацию. 4-5 предложений."""),
        HumanMessage(content=(
            f"Вопрос: {state['question']}\n\n"
            f"Аргументы оптимиста:\n{opt_args}\n\n"
            f"Аргументы пессимиста:\n{pess_args}"
        ))
    ])

    print(f"  [Медиатор] Заключение готово")
    return {"conclusion": response.content}

## Маршрутизация и сборка графа

Условное ребро из узла `pessimist` проверяет счётчик раундов: если прошло 3 раунда, управление передаётся медиатору (`mediate`), иначе — обратно к оптимисту для нового раунда (`debate`). Три раунда — это шесть вызовов LLM для дебатов (по два участника на раунд) плюс один для медиатора.

In [6]:
def should_continue_debate(state: DebateState) -> str:
    """Продолжить дебаты или перейти к синтезу."""
    if state.get("round", 0) >= 3:
        return "mediate"
    return "debate"

# --- Сборка графа ---
graph = StateGraph(DebateState)

graph.add_node("optimist", optimist)
graph.add_node("pessimist", pessimist)
graph.add_node("mediator", mediator)

graph.add_edge(START, "optimist")
graph.add_edge("optimist", "pessimist")
graph.add_conditional_edges("pessimist", should_continue_debate, {
    "debate": "optimist",
    "mediate": "mediator",
})
graph.add_edge("mediator", END)

app = graph.compile()

## Запуск

Запускаем дебаты на тему: "Стоит ли стартапу использовать мультиагентную архитектуру в MVP?" Оптимист и пессимист проведут три раунда спора, после чего медиатор вынесет взвешенное заключение.

In [7]:
result = app.invoke({
    "question": "Стоит ли стартапу использовать мультиагентную архитектуру в MVP?",
    "optimist_args": [],
    "pessimist_args": [],
    "round": 0,
    "conclusion": "",
})

print(f"\nЗаключение медиатора:\n{result['conclusion']}")

  [Оптимист, раунд 1] Да, **иногда стоит**, но **не по умолчанию**.

**Когда мультиагентная архитектур...


  [Пессимист, раунд 1] Скорее **нет, не стоит по умолчанию**. Для MVP мультиагентная архитектура чаще в...


  [Оптимист, раунд 2] Скорее **да — но не по умолчанию, а точечно**. Для MVP мультиагентность стоит бр...


  [Пессимист, раунд 2] Скорее **нет, в MVP мультиагентность обычно не стоит брать**.

1. **Она почти вс...


  [Оптимист, раунд 3] Скорее **да, но только в узких случаях** — и не как “по умолчанию”, а **если мул...


  [Пессимист, раунд 3] Скорее **нет, не стоит по умолчанию**.

1. **MVP нужен для проверки гипотезы, а ...


  [Медиатор] Заключение готово

Заключение медиатора:
Сбалансированно: **по умолчанию — нет, для MVP мультиагентную архитектуру брать не стоит**, потому что она почти всегда усложняет отладку, наблюдаемость и ускорение проверки гипотезы. При этом у оптимиста сильный аргумент в том, что **если мультиагентность сама является частью ценности продукта** — например, распределённый поиск, проверка фактов, автономная кооперация или критичная валидация — тогда откладывать её нельзя, иначе вы тестируете не ту идею. Поэтому практичное правило такое: **начинайте с одного агента + простого оркестратора**, если только не доказано, что разделение ролей напрямую повышает качество, скорость или является ядром дифференциации. Моя рекомендация: **берите мультиагентность в MVP только тогда, когда без неё невозможно проверить ключевую гипотезу или заметно хуже получается основной пользовательский outcome**; во всех остальных случаях отложите её до следующей версии.


## Итог

Паттерн Debate реализует структурированную дискуссию между агентами с противоположными позициями. Ключевые элементы:

- **Раундовая структура**: оптимист и пессимист чередуются в нескольких раундах, каждый раз отвечая на аргументы оппонента
- **Аккумуляция аргументов**: `operator.add` в полях состояния сохраняет полную историю дебатов
- **Медиатор как финальный синтезатор**: получает все аргументы обеих сторон и формирует взвешенное заключение
- **Условная маршрутизация**: счётчик раундов управляет переходом от дебатов к синтезу

Этот паттерн особенно полезен для задач, требующих многогранного анализа: оценка бизнес-решений, анализ рисков, стратегическое планирование.